# Optimus Demo

In [1]:
from IPython.display import clear_output
import pandas as pd
df = pd.read_pickle('demo/data/demo_sample.pkl')
df.head(5)

,kyc_case_id,kyc_create_time,is_failed_kyc,sample_type,ip_geo_ip_static_ip_score,ip_geo_ip_connection_type,ip_geo_ip_user_type,ip_geo_ip_is_anonymous,ip_geo_ip_is_anonymous_proxy,ip_geo_ip_is_anonymous_vpn,ip_geo_ip_is_hosting_provider,ip_geo_ip_is_legitimate_proxy,ip_geo_ip_is_public_proxy,ip_geo_ip_is_residential_proxy,ip_geo_ip_is_tor_exit_node,bw_device_type,bw_device_browser,bw_device_screen_size
4,01b1e219-63bb-4cd0-9fed-d000d1535df3,2025-11-13 09:40:41,0,oot,-990000.000000,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,-990000.0
5,01da0beb-5c70-4c11-a510-b1d1abeb35a0,2025-07-17 03:05:34,0,train,-990000.000000,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,__N.A__,-990000.0
7,01f27f9e-bd0b-439f-ae4b-ec1935cc841d,2025-05-15 07:32:36,0,train,0.400000,Cable/DSL,residential,false,false,false,false,false,false,false,false,Windows,Chrome,2073600.0
9,02a8d1ac-4b94-4dde-ae33-714eafa8a020,2025-12-12 10:53:53,0,oot,-999998.000000,Corporate,hosting,true,false,false,true,false,false,false,false,iPad,Safari,842952.0
17,09a61483-06ee-4fbe-9d25-a3c9e2ed4058,2025-05-25 06:50:43,0,train,47.299999,Cable/DSL,residential,false,false,false,false,false,false,false,false,Linux,Chrome,313960.0


In [2]:
FT_SPEC = {
    "ip_geo_ip_static_ip_score": 'optimal',
    "ip_geo_ip_connection_type": False,
    "ip_geo_ip_user_type": False,
    "ip_geo_ip_is_anonymous": False,
    "ip_geo_ip_is_anonymous_proxy": False,
    "ip_geo_ip_is_anonymous_vpn": False,
    "ip_geo_ip_is_hosting_provider": False,
    "ip_geo_ip_is_legitimate_proxy": False,
    "ip_geo_ip_is_public_proxy": False,
    "ip_geo_ip_is_residential_proxy": False,
    "ip_geo_ip_is_tor_exit_node": False,
    "bw_device_type": False,
    "bw_device_browser": False,
    "bw_device_screen_size": 'optimal',
}
LABEL = "is_failed_kyc"
MISSING_VALUES = ["__N.A__", "__C.N.A__", -990000, -999998, -999999]
FS_PARAMS = {
    'corr_threshold': 0.98,
    'psi_threshold': 0.1,
    'iv_threshold': 0.01,
    'vif_threshold': 10,
    'boosting_select_frac': 1,
    'stability_threshold': 0.05,
}

In [3]:
from optimus.trainer import Train
trainer = Train(
    model_path="demo/models",
    report_path="demo/reports",
    model_type="LR",
    missing_values=MISSING_VALUES,
    tune_method="BO",
    score_floor=0,
    score_cap=1,
    n_bins=10,
    max_evals=10,
    calibration_method="isotonic",
    score_bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    version="4.0.0",
    spec=FT_SPEC,
    ignore_preprocessors=['VIF', 'Boosting', 'Stability'],
    **FS_PARAMS,
)
X = df[FT_SPEC.keys()]
y = df[LABEL]
e = df[df.columns.difference(FT_SPEC.keys()).tolist()]
performance = trainer.fit(X, y, e).transform(X, y, e)

[INFO] Fitting...
[INFO] 1/14 Process ip_geo_ip_static_ip_score
[INFO] 2/14 Process ip_geo_ip_connection_type
[INFO] 3/14 Process ip_geo_ip_user_type
[WARN] ip_geo_ip_user_type have too many categories, suggest to bin before!
[INFO] 4/14 Process ip_geo_ip_is_anonymous
[INFO] 5/14 Process ip_geo_ip_is_anonymous_proxy
[INFO] 6/14 Process ip_geo_ip_is_anonymous_vpn
[INFO] 7/14 Process ip_geo_ip_is_hosting_provider
[INFO] 8/14 Process ip_geo_ip_is_legitimate_proxy
[INFO] 9/14 Process ip_geo_ip_is_public_proxy
[INFO] 10/14 Process ip_geo_ip_is_residential_proxy
[INFO] 11/14 Process ip_geo_ip_is_tor_exit_node
[INFO] 12/14 Process bw_device_type
[INFO] 13/14 Process bw_device_browser
[INFO] 14/14 Process bw_device_screen_size
[INFO] Transforming...
[INFO] 1/14 Process ip_geo_ip_static_ip_score
[INFO] 2/14 Process ip_geo_ip_connection_type
[INFO] 3/14 Process ip_geo_ip_user_type
[INFO] 4/14 Process ip_geo_ip_is_anonymous
[INFO] 5/14 Process ip_geo_ip_is_anonymous_proxy
[INFO] 6/14 Process ip_g

In [9]:
trainer.write_report()

[INFO] Loaded performance from demo/models/20260204_123335/performance
[INFO] Report saved to: demo/reports/model_report_20260204_123335.xlsx
[SUCCESS] Model report written to demo/reports/model_report_20260204_123335.xlsx


'demo/reports/model_report_20260204_123335.xlsx'

In [ ]:
performance = trainer.transform(X, y, e)

trainer.write_report(
    performance=performance,
    report_path='./demo_models',
    report_name=f'model_report_{trainer.ts}'
)

performance['scorecard']['test'].head()
performance['feature_importance'].head(10)

In [ ]:
original_ts = trainer.ts

trainer.refit_model(ts=original_ts, trial_index=5)
new_performance = trainer.transform(X, y, e)
new_performance['scorecard']['test'].head()

In [ ]:
ts = '20260201_210105'

predictor = Train(model_path='./demo_models')
predictions = predictor.transform(X, y, e, ts=ts)